<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 1 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">连接 Doris，查询第一批订单</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">使用统一订单样本，观察 SQL、结果与验收证据。请按顺序运行单元。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 target · Order data · Isolated course database</span>
</div>

By the end of this lab, you will have a running Doris environment, an `events` table containing more than 10 million rows, and analytical results produced from that table. Run the cells in order.

[讲义](course1_introduction_to_apache_doris.md) · [课程入口](../README.md)


## 实验范围

使用独立课程数据库；本 Lab 仅重建 d01_orders。运行前阅读 environments/single-node/README.md。目标是验证连接与正确性，不是测量性能。


### Prepare the single-node environment

已有 Doris：保留默认设置，不启动容器。首次使用 Docker：先阅读[单节点说明](../../environments/single-node/README.md)，在启动 Jupyter 前设置 DW_START_SANDBOX=yes。本步骤只管理课程 02 自己的项目和数据卷。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

import os
from dw_course.docker_runtime import prepare_environment

if os.environ.get("DW_START_SANDBOX") == "yes":
    prepare_environment()
else:
    print("使用已有 Doris；未启动或重启任何容器。")


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = WarehouseLab()




## 1. 记录真实环境

VERSION() 可能是协议兼容版本。对照 FE/BE 的版本与存活信息，不能将开发构建当作正式 4.1.3。


In [ ]:
lab.sql("SELECT VERSION(), @@version_comment");
lab.sql("SHOW FRONTENDS");
lab.sql("SHOW BACKENDS");


## 2. 建表并写入

先使用给定表结构；D03 再讨论为什么选择该模型。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d01_orders")
ddl = order_ddl("d01_orders")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.insert("d01_orders", ORDER_COLUMNS, order_rows(fixture("orders.json")))


## 3. 用业务结果验收

结果必须为 10 笔订单、总金额 1400.00。明细按订单号排序，逐行观察状态和版本。


In [ ]:
lab.sql("SELECT order_id, status, order_amount FROM d01_orders ORDER BY order_id");
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM d01_orders"), [(10, "1400.00")])
lab.close()


## 完成与排查

连接失败先核对 FE 查询端口和认证；写入失败检查 BE Alive 与副本配置。不要为重试而删除整个数据库。下一单元观察这批数据在 Doris 内的组织方式。
